# Perguntas Analíticas - Passos Mágicos Datathon

## Objetivo
Responder 11 perguntas de negócio usando o dataset limpo e unificado.

## Nota sobre Nomenclatura
- **Q = Pergunta**: Cada seção começa com "Qn:" onde n é o número da pergunta
- **Correlação**: Medida de associação entre duas variáveis que varia de -1 a 1
  - Valores próximos a 1: forte relação positiva
  - Valores próximos a -1: forte relação negativa
  - Valores próximos a 0: relação fraca ou nenhuma

## Glossário de Siglas e Convenções de Leitura
- **Sigla + nome completo** na primeira menção de cada indicador.
- **Nomes curtos e técnicos** no código para manter reprodutibilidade.
- **Observações metodológicas explícitas** quando houver limitação de cobertura (ex.: IPP em 2022).

### Indicadores principais
- **IAN**: Indicador de Adequação do Nível
- **IDA**: Indicador de Desempenho Acadêmico
- **IEG**: Indicador de Engajamento
- **IAA**: Indicador de Autoavaliação
- **IPS**: Indicador Psicossocial
- **IPP**: Indicador Psicopedagógico
- **IPV**: Indicador de Ponto de Virada
- **INDE**: Índice de Desenvolvimento Educacional

## Perguntas Analíticas
1. Perfil e evolução do IAN
2. Tendências de desempenho do IDA
3. Correlação IEG-IDA-IPV
4. Alinhamento IAA-IDA
5. Padrões e risco do IPS
6. Validação IAN-IPP
7. Drivers do IPV
8. Combinações de indicadores (IDA + IEG + IPS + IPP) que elevam mais o INDE
9. Sinais precoces de risco
10. Efetividade do programa entre fases
11. Insights adicionais

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuração inicial
project_root = Path.cwd().parent
DATA_DIR = project_root / "data"
OUTPUTS_DIR = project_root / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

# Cores
COLORS = {
    'primary': '#2c3e50',
    'secondary': '#34495e',
    'accent': '#3498db',
    'warning': '#e74c3c',
    'success': '#27ae60',
    'neutral': '#95a5a6'
}

# Configuração dos gráficos
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.edgecolor': COLORS['primary'],
    'grid.color': '#d5d8dc',
    'grid.alpha': 0.3,
    'font.size': 10
})
sns.set_palette([COLORS['primary'], COLORS['accent'], COLORS['secondary'], 
                 COLORS['success'], COLORS['warning']])

# Carregar dados
df = pd.read_csv(DATA_DIR / "dados_unificados.csv")

print(f"Dataset carregado: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"Anos: {df['year'].unique()}")

Dataset carregado: 3030 linhas × 34 colunas
Anos: ['PEDE2022' 'PEDE2023' 'PEDE2024']


## Q1: Perfil do IAN - Adequação Acadêmica

Qual é o perfil geral de deficiência acadêmica (IAN) e como ele evolui?

In [ ]:
# Definir categorias de defasagem com base na escala IAN (0-10)
# IAN mais baixo = maior defasagem
def categorize_deficiency(ian_value):
    if pd.isna(ian_value):
        return 'Desconhecido'
    elif ian_value <= 3.0:
        return 'Severamente Deficiente'
    elif ian_value <= 5.0:
        return 'Moderadamente Deficiente'
    elif ian_value <= 7.0:
        return 'Levemente Deficiente'
    else:
        return 'Adequado'

df['ian_category'] = df['ian'].apply(categorize_deficiency)

# Q1.1: Distribuição geral do IAN
print("="*80)
print("Q1: Perfil do IAN - Adequação Acadêmica")
print("="*80)

print("\n1. Distribuição Geral:")
ian_dist = df['ian_category'].value_counts().sort_index()
ian_dist_pct = (ian_dist / len(df[df['ian_category'] != 'Desconhecido']) * 100).round(1)
for cat in ['Severamente Deficiente', 'Moderadamente Deficiente', 'Levemente Deficiente', 'Adequado']:
    if cat in ian_dist.index:
        print(f"  {cat}: {ian_dist[cat]} estudantes ({ian_dist_pct[cat]}%)")

# Q1.2: Evolução do IAN por ano
print("\n2. Evolução do IAN por Ano:")
for year in ['PEDE2022', 'PEDE2023', 'PEDE2024']:
    df_year = df[df['year'] == year]
    mean_ian = df_year['ian'].mean()
    std_ian = df_year['ian'].std()
    print(f"\n  {year}:")
    print(f"    IAN Médio: {mean_ian:.2f} (±{std_ian:.2f})")
    print(f"    Total de estudantes: {len(df_year)}")
    
    year_dist = df_year['ian_category'].value_counts()
    for cat in ['Severamente Deficiente', 'Moderadamente Deficiente', 'Levemente Deficiente', 'Adequado']:
        if cat in year_dist.index:
            pct = (year_dist[cat] / len(df_year) * 100)
            print(f"    {cat}: {year_dist[cat]} ({pct:.1f}%)")

Q1: Perfil do IAN - Adequação Acadêmica

1. Distribuição Geral:
  Severamente Deficiente: 45 estudantes (1.5%)
  Moderadamente Deficiente: 1642 estudantes (54.2%)
  Adequado: 1343 estudantes (44.3%)

2. Evolução do IAN por Ano:

  PEDE2022:
    IAN Médio: 6.42 (±2.39)
    Total de estudantes: 860
    Severamente Deficiente: 28 (3.3%)
    Moderadamente Deficiente: 573 (66.6%)
    Adequado: 259 (30.1%)

  PEDE2023:
    IAN Médio: 7.24 (±2.54)
    Total de estudantes: 1014
    Severamente Deficiente: 14 (1.4%)
    Moderadamente Deficiente: 538 (53.1%)
    Adequado: 462 (45.6%)

  PEDE2024:
    IAN Médio: 7.68 (±2.50)
    Total de estudantes: 1156
    Severamente Deficiente: 3 (0.3%)
    Moderadamente Deficiente: 531 (45.9%)
    Adequado: 622 (53.8%)


## Q2: Desempenho IDA - Tendências de Performance Acadêmica

In [3]:
print("\n" + "="*80)
print("Q2: Desempenho IDA - Tendências de Performance Acadêmica")
print("="*80)

print("\n1. IDA Médio por Ano:")
ida_by_year = df.groupby('year')['ida'].agg(['count', 'mean', 'std', 'min', 'max']).round(2)
print(ida_by_year)

print("\n2. Mudança Ano a Ano:")
years = ['PEDE2022', 'PEDE2023', 'PEDE2024']
ida_means = [df[df['year'] == year]['ida'].mean() for year in years]
for i in range(1, len(years)):
    change = ida_means[i] - ida_means[i-1]
    change_pct = (change / ida_means[i-1] * 100) if ida_means[i-1] != 0 else 0
    direction = "↑" if change > 0 else "↓"
    print(f"  {years[i-1]} → {years[i]}: {direction} {abs(change):.2f} ({change_pct:+.1f}%)")

print("\n3. Distribuição IDA por Fase (2024):")
df_2024 = df[df['year'] == 'PEDE2024']
ida_by_phase = df_2024.groupby('phase')['ida'].agg(['count', 'mean', 'std']).round(2)
print(ida_by_phase)


Q2: Desempenho IDA - Tendências de Performance Acadêmica

1. IDA Médio por Ano:
          count  mean   std  min   max
year                                  
PEDE2022    860  6.09  2.05  0.0   9.9
PEDE2023    937  6.66  1.60  0.0  10.0
PEDE2024   1055  6.35  2.13  0.0  10.0

2. Mudança Ano a Ano:
  PEDE2022 → PEDE2023: ↑ 0.57 (+9.4%)
  PEDE2023 → PEDE2024: ↓ 0.31 (-4.7%)

3. Distribuição IDA por Fase (2024):
       count  mean   std
phase                   
1A        14  7.57  1.09
1B        15  7.18  1.60
1C        14  7.29  2.00
1D        10  6.60  2.24
1E         8  6.31  2.70
...      ...   ...   ...
8D         0   NaN   NaN
8E         0   NaN   NaN
8F         1  8.00   NaN
9          0   NaN   NaN
ALFA     196  7.32  1.94

[72 rows x 3 columns]


## Q3: Correlação IEG-IDA-IPV - Impacto do Engajamento

In [ ]:
print("\n" + "="*80)
print("Q3: Correlação IEG-IDA-IPV - Impacto do Engajamento")
print("="*80)

# Remover linhas com indicadores centrais faltantes
df_complete = df.dropna(subset=['ieg', 'ida', 'ipv'])

print(f"\nAnalisando {len(df_complete)} registros completos")

# Matriz de correlação
corr_matrix = df_complete[['ieg', 'ida', 'ipv']].corr().round(3)

print("\n1. Matriz de Correlação:")
print(corr_matrix)

print("\n2. Interpretação:")
ieg_ida_corr = corr_matrix.loc['ieg', 'ida']
print(f"  IEG (Engajamento) vs IDA (Desempenho): r = {ieg_ida_corr:.3f}")
if abs(ieg_ida_corr) > 0.5:
    print(f"    → Relação FORTE: Quanto maior o engajamento, maior o desempenho acadêmico")
elif abs(ieg_ida_corr) > 0.3:
    print(f"    → Relação MODERADA: Engajamento influencia moderadamente o desempenho")
else:
    print(f"    → Relação FRACA: Engajamento tem pouca influência no desempenho isoladamente")

ieg_ipv_corr = corr_matrix.loc['ieg', 'ipv']
print(f"\n  IEG (Engajamento) vs IPV (Ponto de Virada): r = {ieg_ipv_corr:.3f}")
if abs(ieg_ipv_corr) > 0.5:
    print(f"    → Relação FORTE: Estudantes engajados têm maior probabilidade de viradas")
elif abs(ieg_ipv_corr) > 0.3:
    print(f"    → Relação MODERADA: Engajamento influencia moderadamente as viradas")
else:
    print(f"    → Relação FRACA: Engajamento sozinho não prevê viradas")

ida_ipv_corr = corr_matrix.loc['ida', 'ipv']
print(f"\n  IDA (Desempenho) vs IPV (Ponto de Virada): r = {ida_ipv_corr:.3f}")
if abs(ida_ipv_corr) > 0.5:
    print(f"    → Relação FORTE: Estudantes com melhor desempenho têm mais viradas")
elif abs(ida_ipv_corr) > 0.3:
    print(f"    → Relação MODERADA: Desempenho influencia moderadamente as viradas")
else:
    print(f"    → Relação FRACA: Desempenho sozinho não prevê viradas")


Q3: Correlação IEG-IDA-IPV - Impacto do Engajamento

Analisando 2851 registros completos

1. Matriz de Correlação:
       ieg    ida    ipv
ieg  1.000  0.543  0.558
ida  0.543  1.000  0.557
ipv  0.558  0.557  1.000

2. Interpretação:
  Correlação IEG vs IDA: 0.543
    → Relação FORTE: engajamento impacta diretamente o desempenho

  Correlação IEG vs IPV: 0.558
    → Relação FORTE: engajamento prediz pontos de virada

  Correlação IDA vs IPV: 0.557


## Q4: Alinhamento IAA-IDA - Autoavaliação vs. Realidade

In [ ]:
print("\n" + "="*80)
print("Q4: Alinhamento IAA-IDA - Autoavaliação vs Realidade")
print("="*80)

df_alignment = df.dropna(subset=['iaa', 'ida'])

# Calcular lacuna de percepção (Autoavaliação - Desempenho real)
df_alignment['perception_gap'] = df_alignment['iaa'] - df_alignment['ida']

print(f"\nAnalisando {len(df_alignment)} registros com IAA e IDA")

print("\n1. Análise de Alinhamento:")
print(f"  IAA médio: {df_alignment['iaa'].mean():.2f}")
print(f"  IDA médio: {df_alignment['ida'].mean():.2f}")
print(f"  Gap (IAA - IDA): {df_alignment['perception_gap'].mean():.2f}")

# Classificar percepção
overestimate = (df_alignment['perception_gap'] > 1.0).sum()
aligned = ((df_alignment['perception_gap'] >= -1.0) & (df_alignment['perception_gap'] <= 1.0)).sum()
underestimate = (df_alignment['perception_gap'] < -1.0).sum()

print("\n2. Calibração da Percepção:")
print(f"  Superestimam (IAA > IDA em 1+): {overestimate} ({overestimate/len(df_alignment)*100:.1f}%)")
print(f"  Bem alinhados (±1 ponto): {aligned} ({aligned/len(df_alignment)*100:.1f}%)")
print(f"  Subestimam (IDA > IAA em 1+): {underestimate} ({underestimate/len(df_alignment)*100:.1f}%)")

print("\n3. Correlação:")
corr_iaa_ida = df_alignment['iaa'].corr(df_alignment['ida'])
print(f"  Correlação IAA-IDA: {corr_iaa_ida:.3f}")


Q4: Alinhamento IAA-IDA - Autoavaliação vs Realidade

Analisando 2851 registros com IAA e IDA

1. Análise de Alinhamento:
  IAA médio: 7.93
  IDA médio: 6.38
  Gap (IAA - IDA): 1.55

2. Calibração da Percepção:
  Superestimam (IAA > IDA em 1+): 1863 (65.3%)
  Bem alinhados (±1 ponto): 660 (23.1%)
  Subestimam (IDA > IAA em 1+): 328 (11.5%)

3. Correlação:
  Correlação IAA-IDA: 0.115


## Q5: Padrões de IPS - Sinais de Risco Psicossocial

In [ ]:
print("\n" + "="*80)
print("Q5: Padrões de IPS - Sinais de Risco Psicossocial")
print("="*80)

df_ips = df.dropna(subset=['ips', 'ida'])

print(f"\nAnalisando {len(df_ips)} registros com dados de IPS")

# Categorizar IPS (Indicador Psicossocial)
def categorize_ips(ips_value):
    if pd.isna(ips_value):
        return 'Desconhecido'
    elif ips_value <= 4.0:
        return 'Risco Alto'
    elif ips_value <= 6.0:
        return 'Risco Moderado'
    else:
        return 'Risco Baixo'

df_ips['ips_category'] = df_ips['ips'].apply(categorize_ips)

print("\n1. Distribuição de Risco IPS:")
ips_dist = df_ips['ips_category'].value_counts()
for cat in ['Risco Alto', 'Risco Moderado', 'Risco Baixo']:
    if cat in ips_dist.index:
        pct = (ips_dist[cat] / len(df_ips) * 100)
        print(f"  {cat}: {ips_dist[cat]} estudantes ({pct:.1f}%)")

print("\n2. IPS vs Desempenho IDA:")
for cat in ['Risco Alto', 'Risco Moderado', 'Risco Baixo']:
    df_cat = df_ips[df_ips['ips_category'] == cat]
    if len(df_cat) > 0:
        print(f"  {cat}: IDA médio = {df_cat['ida'].mean():.2f} (n={len(df_cat)})")

print("\n3. Correlação IPS-IDA:")
corr_ips_ida = df_ips['ips'].corr(df_ips['ida'])
print(f"  Correlação: {corr_ips_ida:.3f}")


Q5: Padrões de IPS - Sinais de Risco Psicossocial

Analisando 2845 registros com dados de IPS

1. Distribuição de Risco IPS:
  Risco Alto: 444 estudantes (15.6%)
  Risco Moderado: 482 estudantes (16.9%)
  Risco Baixo: 1919 estudantes (67.5%)

2. IPS vs Desempenho IDA:
  Risco Alto: IDA médio = 6.51 (n=444)
  Risco Moderado: IDA médio = 6.04 (n=482)
  Risco Baixo: IDA médio = 6.43 (n=1919)

3. Correlação IPS-IDA:
  Correlação: 0.022


## Q6: Validação IAN-IPP - Consistência da Avaliação de Deficiência

In [4]:
print("\n" + "="*80)
print("Q6: Validação IAN-IPP - Consistência da Avaliação de Deficiência")
print("="*80)

df_validation = df.dropna(subset=['ian', 'ipp'])

print(f"\nAnalisando {len(df_validation)} registros com IAN e IPP")
print("Observação metodológica: IPP não está disponível em 2022 na base original; esta análise reflete majoritariamente 2023-2024.")

if 'year' in df_validation.columns:
    print("\nDistribuição da amostra por ano (Q6):")
    print(df_validation['year'].value_counts().sort_index())

print("\n1. Estatísticas Sumárias:")
print(f"  IAN médio: {df_validation['ian'].mean():.2f} (dp: {df_validation['ian'].std():.2f})")
print(f"  IPP médio: {df_validation['ipp'].mean():.2f} (dp: {df_validation['ipp'].std():.2f})")

print("\n2. Verificação de Consistência:")
corr_ian_ipp = df_validation['ian'].corr(df_validation['ipp'])
print(f"  Correlação IAN-IPP: {corr_ian_ipp:.3f}")

if abs(corr_ian_ipp) > 0.5:
    print(f"    → Concordância FORTE: avaliações são consistentes")
elif abs(corr_ian_ipp) > 0.3:
    print(f"    → Concordância MODERADA")
else:
    print(f"    → Concordância FRACA: avaliações podem diferir")

# Análise da lacuna
df_validation['ian_ipp_gap'] = df_validation['ian'] - df_validation['ipp']
print(f"\n3. Gap de Avaliação (IAN - IPP):")
print(f"  Gap médio: {df_validation['ian_ipp_gap'].mean():.2f}")
print(f"  Casos onde IAN > IPP (mais deficiente por IAN): {(df_validation['ian_ipp_gap'] > 0.5).sum()}")
print(f"  Casos onde IPP > IAN (mais deficiente por IPP): {(df_validation['ian_ipp_gap'] < -0.5).sum()}")


Q6: Validação IAN-IPP - Consistência da Avaliação de Deficiência

Analisando 1992 registros com IAN e IPP
Observação metodológica: IPP não está disponível em 2022 na base original; esta análise reflete majoritariamente 2023-2024.

Distribuição da amostra por ano (Q6):
year
PEDE2023     938
PEDE2024    1054
Name: count, dtype: int64

1. Estatísticas Sumárias:
  IAN médio: 7.26 (dp: 2.52)
  IPP médio: 7.56 (dp: 0.94)

2. Verificação de Consistência:
  Correlação IAN-IPP: 0.123
    → Concordância FRACA: avaliações podem diferir

3. Gap de Avaliação (IAN - IPP):
  Gap médio: -0.30
  Casos onde IAN > IPP (mais deficiente por IAN): 906
  Casos onde IPP > IAN (mais deficiente por IPP): 1048


## Q7: Drivers de IPV - Preditores de Ponto de Virada

In [ ]:
print("\n" + "="*80)
print("Q7: Drivers de IPV - Preditores de Ponto de Virada")
print("="*80)

# Incluir todos os preditores potenciais do IPV
ipv_predictors = df.dropna(subset=['ipv', 'ian', 'ida', 'ieg', 'iaa', 'ips'])

print(f"\nAnalisando {len(ipv_predictors)} registros com dados completos")

print("\n1. Correlação com IPV:")
ipv_correlations = {}
for col in ['ian', 'ida', 'ieg', 'iaa', 'ips']:
    corr = ipv_predictors[col].corr(ipv_predictors['ipv'])
    ipv_correlations[col] = corr
    print(f"  {col.upper()} vs IPV: {corr:.3f}")

# Ordenar por importância (valor absoluto da correlação)
sorted_correlations = sorted(ipv_correlations.items(), key=lambda x: abs(x[1]), reverse=True)

print("\n2. Ranking de Influência nos Pontos de Virada:")
for rank, (indicator, corr) in enumerate(sorted_correlations, 1):
    strength = "Forte" if abs(corr) > 0.5 else "Moderada" if abs(corr) > 0.3 else "Fraca"
    print(f"  {rank}. {indicator.upper()}: {corr:.3f} ({strength})")


Q7: Drivers de IPV - Preditores de Ponto de Virada

Analisando 2845 registros com dados completos

1. Correlação com IPV:
  IAN vs IPV: 0.149
  IDA vs IPV: 0.557
  IEG vs IPV: 0.558
  IAA vs IPV: 0.062
  IPS vs IPV: -0.049

2. Ranking de Influência nos Pontos de Virada:
  1. IEG: 0.558 (Forte)
  2. IDA: 0.557 (Forte)
  3. IAN: 0.149 (Fraca)
  4. IAA: 0.062 (Fraca)
  5. IPS: -0.049 (Fraca)


## Q8: Multidimensionalidade dos Indicadores - Combinações que Elevam o INDE

In [5]:
print("\n" + "="*80)
print("Q8: Multidimensionalidade dos Indicadores - Combinações que Elevam o INDE")
print("="*80)

from itertools import combinations

# Variável-alvo: INDE (média das colunas disponíveis)
inde_cols = [col for col in df.columns if 'inde' in col.lower()]
for col in inde_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['inde_combined'] = df[inde_cols].astype(float).mean(axis=1)

# Indicadores solicitados no novo enunciado
q8_indicators = ['ida', 'ieg', 'ips', 'ipp']

# Base para análise (com IPP disponível)
q8_base = df.dropna(subset=q8_indicators + ['inde_combined']).copy()
print(f"\nRegistros válidos para Q8: {len(q8_base)}")
print(f"Indicadores analisados: {q8_indicators}")
print("Observação metodológica: como IPP não existe em 2022 na base original, as combinações com IPP refletem principalmente 2023-2024.")

if 'year' in q8_base.columns:
    print("\nDistribuição da amostra por ano (Q8):")
    print(q8_base['year'].value_counts().sort_index())

# Padronizar indicadores para comparar combinações na mesma escala
for col in q8_indicators:
    std = q8_base[col].std()
    if std == 0 or pd.isna(std):
        q8_base[f'z_{col}'] = 0
    else:
        q8_base[f'z_{col}'] = (q8_base[col] - q8_base[col].mean()) / std

print("\n1. Uplift de INDE por combinação de indicadores")
combo_results = []

for k in [2, 3, 4]:
    for combo in combinations(q8_indicators, k):
        z_cols = [f'z_{c}' for c in combo]
        score_name = f'score_{"_".join(combo)}'
        q8_base[score_name] = q8_base[z_cols].mean(axis=1)

        q_top = q8_base[score_name].quantile(0.75)
        q_bottom = q8_base[score_name].quantile(0.25)

        top_group = q8_base[q8_base[score_name] >= q_top]
        bottom_group = q8_base[q8_base[score_name] <= q_bottom]

        top_inde = top_group['inde_combined'].mean()
        bottom_inde = bottom_group['inde_combined'].mean()
        uplift = top_inde - bottom_inde

        combo_results.append({
            'combination': ' + '.join([c.upper() for c in combo]),
            'n_indicators': k,
            'inde_top25': top_inde,
            'inde_bottom25': bottom_inde,
            'uplift_inde': uplift,
            'corr_score_inde': q8_base[score_name].corr(q8_base['inde_combined'])
        })

combo_df = pd.DataFrame(combo_results).sort_values('uplift_inde', ascending=False).reset_index(drop=True)

print(combo_df[['combination', 'n_indicators', 'uplift_inde', 'corr_score_inde']].head(10).round(3))

print("\n2. Top 3 combinações que mais elevam o INDE")
for i, row in combo_df.head(3).iterrows():
    print(f"  {i+1}. {row['combination']} | Uplift INDE: {row['uplift_inde']:.3f} | Corr: {row['corr_score_inde']:.3f}")

print("\n3. Evidência da combinação completa (IDA + IEG + IPS + IPP)")
full_combo = combo_df[combo_df['combination'] == 'IDA + IEG + IPS + IPP']
if not full_combo.empty:
    r = full_combo.iloc[0]
    print(f"  Uplift INDE (top25 vs bottom25): {r['uplift_inde']:.3f}")
    print(f"  Correlação score combinado vs INDE: {r['corr_score_inde']:.3f}")
else:
    print("  Combinação completa não pôde ser estimada por insuficiência de dados.")


Q8: Multidimensionalidade dos Indicadores - Combinações que Elevam o INDE

Registros válidos para Q8: 1624
Indicadores analisados: ['ida', 'ieg', 'ips', 'ipp']
Observação metodológica: como IPP não existe em 2022 na base original, as combinações com IPP refletem principalmente 2023-2024.

Distribuição da amostra por ano (Q8):
year
PEDE2023     570
PEDE2024    1054
Name: count, dtype: int64

1. Uplift de INDE por combinação de indicadores
             combination  n_indicators  uplift_inde  corr_score_inde
0              IDA + IEG             2        1.713            0.753
1        IDA + IEG + IPP             3        1.690            0.762
2  IDA + IEG + IPS + IPP             4        1.680            0.748
3        IDA + IEG + IPS             3        1.666            0.724
4              IDA + IPP             2        1.510            0.689
5              IEG + IPP             2        1.504            0.688
6        IDA + IPS + IPP             3        1.477            0.646
7    

## Q9: Detecção Precoce de Risco - Análise de Padrões

In [ ]:
print("\n" + "="*80)
print("Q9: Detecção Precoce de Risco - Análise de Padrões")
print("="*80)

# Definir estudantes em risco: baixo em IAN, IDA ou engajamento
df_risk = df.copy()

# Critérios de risco
df_risk['at_risk'] = 0
df_risk.loc[df_risk['ian'] <= 3.0, 'at_risk'] += 1  # Severamente deficiente
df_risk.loc[df_risk['ida'] <= 4.0, 'at_risk'] += 1  # Desempenho acadêmico baixo
df_risk.loc[df_risk['ieg'] <= 4.0, 'at_risk'] += 1  # Engajamento baixo
df_risk.loc[df_risk['ips'] <= 4.0, 'at_risk'] += 1  # Risco psicossocial alto

# Classificação do risco
df_risk['risk_level'] = pd.cut(df_risk['at_risk'], bins=[-1, 0, 1, 2, 5], 
                                labels=['Sem Risco', 'Risco Baixo', 'Risco Moderado', 'Risco Alto'])

print("\n1. Distribuição de Nível de Risco:")
risk_dist = df_risk['risk_level'].value_counts()
for risk in ['Risco Alto', 'Risco Moderado', 'Risco Baixo', 'Sem Risco']:
    if risk in risk_dist.index:
        count = risk_dist[risk]
        pct = (count / len(df_risk[df_risk['risk_level'].notna()]) * 100)
        print(f"  {risk}: {count} estudantes ({pct:.1f}%)")

print("\n2. Perfil de Alto Risco:")
df_high_risk = df_risk[df_risk['risk_level'] == 'Risco Alto']
if len(df_high_risk) > 0:
    print(f"  Total: {len(df_high_risk)} estudantes")
    print(f"  IAN médio: {df_high_risk['ian'].mean():.2f}")
    print(f"  IDA médio: {df_high_risk['ida'].mean():.2f}")
    print(f"  IEG médio: {df_high_risk['ieg'].mean():.2f}")
    print(f"  IPS médio: {df_high_risk['ips'].mean():.2f}")

print("\n3. Risco por Ano:")
for year in ['PEDE2022', 'PEDE2023', 'PEDE2024']:
    df_year_risk = df_risk[df_risk['year'] == year]
    high_risk_pct = (df_year_risk['risk_level'] == 'Risco Alto').sum() / len(df_year_risk) * 100
    print(f"  {year}: {high_risk_pct:.1f}% em alto risco")


Q9: Detecção Precoce de Risco - Análise de Padrões

1. Distribuição de Nível de Risco:
  Risco Alto: 10 estudantes (0.3%)
  Risco Moderado: 95 estudantes (3.1%)
  Risco Baixo: 829 estudantes (27.4%)
  Sem Risco: 2096 estudantes (69.2%)

2. Perfil de Alto Risco:
  Total: 10 estudantes
  IAN médio: 3.25
  IDA médio: 2.11
  IEG médio: 5.07
  IPS médio: 4.39

3. Risco por Ano:
  PEDE2022: 0.6% em alto risco
  PEDE2023: 0.4% em alto risco
  PEDE2024: 0.1% em alto risco


## Q10: Efetividade do Programa - Melhorias entre Fases

In [11]:
print("\n" + "="*80)
print("Q10: Efetividade do Programa - Melhorias entre Fases")
print("="*80)

print("\n1. Desempenho por Fase (coorte 2024):")
df_2024 = df[df['year'] == 'PEDE2024'].dropna(subset=['phase', 'ian', 'ida'])

if len(df_2024) > 0:
    phase_performance = df_2024.groupby('phase')[['ian', 'ida', 'ieg', 'iaa', 'ips', 'ipv']].agg(['count', 'mean']).round(2)
    print("\nIndicadores Médios por Fase:")
    for phase in sorted(df_2024['phase'].unique()):
        if pd.notna(phase):
            df_phase = df_2024[df_2024['phase'] == phase]
            print(f"\n  Fase {phase} (n={len(df_phase)}):")
            print(f"    IAN: {df_phase['ian'].mean():.2f} (Adequação)")
            print(f"    IDA: {df_phase['ida'].mean():.2f} (Desempenho)")
            print(f"    IEG: {df_phase['ieg'].mean():.2f} (Engajamento)")
            print(f"    IAA: {df_phase['iaa'].mean():.2f} (Autoavaliação)")

print("\n2. Evolução da Coorte Ano a Ano:")
print("\n(Comparando IDA médio entre anos)")
for year in ['PEDE2022', 'PEDE2023', 'PEDE2024']:
    df_year = df[df['year'] == year]
    print(f"  {year}: IDA médio = {df_year['ida'].mean():.2f}")


Q10: Efetividade do Programa - Melhorias entre Fases

1. Desempenho por Fase (coorte 2024):

Indicadores Médios por Fase:

  Fase 1A (n=14):
    IAN: 6.07 (Adequação)
    IDA: 7.57 (Desempenho)
    IEG: 8.88 (Engajamento)
    IAA: 9.06 (Autoavaliação)

  Fase 1B (n=15):
    IAN: 7.00 (Adequação)
    IDA: 7.18 (Desempenho)
    IEG: 8.66 (Engajamento)
    IAA: 8.37 (Autoavaliação)

  Fase 1C (n=14):
    IAN: 6.79 (Adequação)
    IDA: 7.29 (Desempenho)
    IEG: 9.24 (Engajamento)
    IAA: 9.10 (Autoavaliação)

  Fase 1D (n=10):
    IAN: 7.00 (Adequação)
    IDA: 6.60 (Desempenho)
    IEG: 8.48 (Engajamento)
    IAA: 9.05 (Autoavaliação)

  Fase 1E (n=8):
    IAN: 6.25 (Adequação)
    IDA: 6.31 (Desempenho)
    IEG: 7.91 (Engajamento)
    IAA: 9.24 (Autoavaliação)

  Fase 1G (n=16):
    IAN: 7.50 (Adequação)
    IDA: 7.27 (Desempenho)
    IEG: 8.92 (Engajamento)
    IAA: 8.69 (Autoavaliação)

  Fase 1H (n=12):
    IAN: 5.42 (Adequação)
    IDA: 5.79 (Desempenho)
    IEG: 7.01 (Engajamento

## Q11: Insights Adicionais

In [12]:
print("\n" + "="*80)
print("Q11: Insights Adicionais - Descobertas Guiadas por Dados")
print("="*80)

print("\n1. Desempenho por Matéria Acadêmica (Matemática, Português, Inglês):")
df_subjects = df.dropna(subset=['math', 'portuguese'])
if len(df_subjects) > 0:
    print(f"\n  Matemática: {df_subjects['math'].mean():.2f} (±{df_subjects['math'].std():.2f})")
    print(f"  Português: {df_subjects['portuguese'].mean():.2f} (±{df_subjects['portuguese'].std():.2f})")
    if 'english' in df.columns:
        df_english = df[df['english'].notna()]
        if len(df_english) > 0:
            print(f"  Inglês: {df_english['english'].mean():.2f} (±{df_english['english'].std():.2f}) - {len(df_english)} estudantes")

print("\n2. Distribuição e Desempenho por Gênero:")
if 'gender' in df.columns:
    df_gender = df.dropna(subset=['gender', 'ida'])
    if len(df_gender) > 0:
        gender_perf = df_gender.groupby('gender')['ida'].agg(['count', 'mean']).round(2)
        print(gender_perf)

print("\n3. Análise de Idade:")
age_cols = [col for col in df.columns if 'age' in col.lower() or 'defasagem' in col.lower()]
if age_cols:
    print(f"  Colunas de idade: {age_cols}")
    for col in age_cols:
        df_age = df[df[col].notna()]
        if len(df_age) > 0:
            numeric_age = pd.to_numeric(df_age[col], errors='coerce')
            mean_age = numeric_age.mean()
            if pd.notna(mean_age):
                print(f"  {col} média: {mean_age:.1f} anos")
            else:
                print(f"  {col} média: n/d (dados não numéricos)")

print("\n4. Estatísticas Gerais:")
print(f"  Total de estudantes analisados: {len(df)}")
print(f"  Período: 2022-2024 (3 anos)")
print(f"  Dados faltantes (média): {df.isnull().sum().sum() / (len(df) * df.shape[1]) * 100:.1f}%")


Q11: Insights Adicionais - Descobertas Guiadas por Dados

1. Desempenho por Matéria Acadêmica (Matemática, Português, Inglês):

  Matemática: 6.17 (±2.40)
  Português: 6.43 (±2.14)
  Inglês: 6.29 (±2.73) - 1091 estudantes

2. Distribuição e Desempenho por Gênero:
           count  mean
gender                
Feminino    1066  6.49
Masculino    926  6.51
Menina       457  6.18
Menino       403  6.00

3. Análise de Idade:
  Colunas de idade: ['age_2022', 'Defasagem', 'age']
  age_2022 média: 12.1 anos
  Defasagem média: -0.5 anos
  age média: 12.7 anos

4. Estatísticas Gerais:
  Total de estudantes analisados: 3030
  Período: 2022-2024 (3 anos)
  Dados faltantes (média): 30.7%


## Resumo

As 11 perguntas analíticas foram respondidas com achados guiados por dados.